## Data Generator

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

print("Generating IoT sensor datasets...")
print("="*60)

# Configuration - REDUCED SIZE for faster processing
n_devices = 1000
n_readings = 1000000

print(f"\nConfiguration:")
print(f"- Devices: {n_devices:,}")
print(f"- Readings: {n_readings:,}")
print(f"- Expected size: ~100MB\n")

# Generate devices
print("Step 1/3: Generating devices...")
device_types = ['temperature', 'humidity', 'pressure', 'multi']
manufacturers = ['SensorCorp', 'IoTech', 'DataSense']

devices = pd.DataFrame({
    'device_id': [f'DEV_{i:05d}' for i in range(n_devices)],
    'device_type': np.random.choice(device_types, n_devices),
    'manufacturer': np.random.choice(manufacturers, n_devices),
    'installation_date': [datetime(2022,1,1) + timedelta(days=int(x)) 
                          for x in np.random.randint(0, 730, n_devices)],
    'max_temp': np.random.uniform(70, 100, n_devices),
    'min_temp': np.random.uniform(-10, 10, n_devices)
})
devices.to_csv('devices.csv', index=False)
print(f"✓ Devices saved: {len(devices):,} records\n")

# Generate sensor readings with data skew
print("Step 2/3: Generating sensor readings...")

# Power law distribution - some devices more active (data skew)
device_activity = np.random.power(0.3, n_readings)
device_indices = (device_activity * (n_devices - 1)).astype(int)
selected_devices = devices.iloc[device_indices]['device_id'].values

# Create timestamps
timestamps = [datetime(2023,1,1) + timedelta(seconds=int(x)) 
              for x in np.random.randint(0, 365*24*3600, n_readings)]

readings = pd.DataFrame({
    'device_id': selected_devices,
    'timestamp': timestamps,
    'temperature': np.random.normal(25, 15, n_readings),
    'humidity': np.random.normal(60, 20, n_readings),
    'pressure': np.random.normal(1013, 25, n_readings),
    'battery_level': np.random.uniform(0, 100, n_readings),
    'status_code': np.random.choice(['OK', 'WARNING', 'ERROR'], 
                                   n_readings, p=[0.90, 0.08, 0.02])
})

print(f"✓ Generated {len(readings):,} readings\n")

# Introduce data quality issues
print("Step 3/3: Adding data quality issues...")

# 1. Missing temperatures (2%)
null_idx = np.random.choice(readings.index, int(0.02*n_readings), replace=False)
readings.loc[null_idx, 'temperature'] = None
print(f"✓ Added {len(null_idx):,} missing temperatures")

# 2. Corrupted timestamps (0.5%) - FIXED: Convert to string first
corrupt_idx = np.random.choice(readings.index, int(0.005*n_readings), replace=False)
# Convert timestamp column to object type to allow mixed types
readings['timestamp'] = readings['timestamp'].astype(object)
readings.loc[corrupt_idx, 'timestamp'] = 'CORRUPTED'
print(f"✓ Added {len(corrupt_idx):,} corrupted timestamps")

# Save to CSV
print("\nSaving to CSV...")
readings.to_csv('sensor_readings.csv', index=False)

print("\n" + "="*60)
print("✅ Data generation complete!")
print("="*60)
print(f"\nGenerated files:")
print(f"- devices.csv: {len(devices):,} records (~100KB)")
print(f"- sensor_readings.csv: {len(readings):,} records (~100MB)")
print(f"\nData quality issues:")
print(f"- Missing temperatures: {len(null_idx):,} ({len(null_idx)/len(readings)*100:.1f}%)")
print(f"- Corrupted timestamps: {len(corrupt_idx):,} ({len(corrupt_idx)/len(readings)*100:.1f}%)")
print(f"\nYou can now run your Spark exercise!")

Generating IoT sensor datasets...

Configuration:
- Devices: 1,000
- Readings: 1,000,000
- Expected size: ~100MB

Step 1/3: Generating devices...
✓ Devices saved: 1,000 records

Step 2/3: Generating sensor readings...
✓ Generated 1,000,000 readings

Step 3/3: Adding data quality issues...
✓ Added 20,000 missing temperatures
✓ Added 5,000 corrupted timestamps

Saving to CSV...

✅ Data generation complete!

Generated files:
- devices.csv: 1,000 records (~100KB)
- sensor_readings.csv: 1,000,000 records (~100MB)

Data quality issues:
- Missing temperatures: 20,000 (2.0%)
- Corrupted timestamps: 5,000 (0.5%)

You can now run your Spark exercise!


## SparkSession Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

# --- SparkSession Configuration ---
spark = SparkSession.builder \
    .appName("IoT_Sensor_Analysis_Project") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("="*50)
print("Task 1.1: SparkSession Configured Successfully")
print("="*50)

# (Active Configurations)
print("\n--- Active Spark Configurations ---")
configurations = sc.getConf().getAll()
for conf in configurations:
    print(f"{conf[0]}: {conf[1]}")

# --- Cluster Information ---
print("\n--- Cluster Information ---")
print(f"Spark Version: {spark.version}")
print(f"Master: {sc.master}")
print(f"Web UI URL: {sc.uiWebUrl}")
print(f"Default Parallelism: {sc.defaultParallelism}")
print(f"Spark User: {sc.sparkUser()}")

# Accumulator Example
print("\n--- Accumulator Example ---")
error_accumulator = sc.accumulator(0)

data = [1, 2, -1, 4, -5, 6]
rdd = sc.parallelize(data)

def count_errors(x):
    global error_accumulator
    if x < 0:
        error_accumulator += 1

rdd.foreach(count_errors)

print(f"Sample Data: {data}")
print("Logic: Negative numbers are considered errors.")
print(f"Total Errors Found (Accumulator Value): {error_accumulator.value}")

print("\n" + "="*50)